# Case Study: End-to-End ML Project

## Regression: Energy Consumption Forecasting

### Problem Statement and Business Context

Accurate energy consumption forecasting is critical for power grid management, energy trading, and sustainability planning. In this case study, we'll build a model to predict hourly energy consumption for a commercial building, helping facility managers optimize energy usage and reduce costs.

### Setup and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('fivethirtyeight')
sns.set_palette('viridis')

In [ ]:
# Define a function to generate synthetic data if needed
def generate_synthetic_energy_data(start_date='2020-01-01', end_date='2022-12-31', 
                                  random_seed=42):
    """
    Generate synthetic building energy consumption data
    
    Parameters:
    - start_date: Beginning date for the dataset (string)
    - end_date: End date for the dataset (string)
    - random_seed: Seed for reproducibility
    
    Returns:
    - DataFrame with hourly energy consumption data
    """
    # Set random seed for reproducibility
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    # Create date range with hourly frequency
    # date_range = pd.date_range(start=start_date, end=end_date, freq='H')
    date_range = pd.date_range(start=start_date, end=end_date, freq='h')
    n_samples = len(date_range)
    
    # Initialize dataframe
    df = pd.DataFrame(index=date_range)
    df.index.name = 'timestamp'
    
    # Add temporal features first (we'll use these to generate other features)
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['month'] = df.index.month
    df['day_of_week'] = df.index.dayofweek  # 0=Monday, 6=Sunday
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Generate realistic outdoor temperature with seasonal patterns
    # Base seasonal pattern
    seasonal_temp = 15 - 15 * np.cos(2 * np.pi * (df.index.dayofyear / 365))
    
    # Add daily variations (warmer during day, cooler at night)
    hour_temp = 5 * np.sin(np.pi * (df['hour'] - 4) / 24)
    
    # Add some random noise
    temp_noise = np.random.normal(0, 2, n_samples)
    
    # Combine components
    df['outdoor_temperature'] = seasonal_temp + hour_temp + temp_noise
    
    # Generate energy consumption with realistic patterns
    
    # Base load (always present)
    base_load = 20 + np.random.normal(0, 2, n_samples)
    
    # Business hours load (higher during working hours on weekdays)
    business_mask = ((df['hour'] >= 8) & (df['hour'] <= 18) & (df['day_of_week'] < 5))
    business_load = np.zeros(n_samples)
    business_load[business_mask] = 30 + np.random.normal(0, 5, sum(business_mask))
    
    # Temperature-dependent load (HVAC)
    # Heating (when cold)
    heating_load = np.maximum(0, (18.5 - df['outdoor_temperature']) * 2)
    
    # Cooling (when hot)
    cooling_load = np.maximum(0, (df['outdoor_temperature'] - 21) * 3)
    
    # Seasonal adjustments (more energy use in winter months)
    seasonal_factor = 1.2 + 0.3 * np.cos(2 * np.pi * (df.index.dayofyear / 365))
    
    # Time of day factor (varies throughout day)
    time_factor = 1 + 0.5 * np.sin(np.pi * (df['hour'] - 2) / 12)
    
    # Weekend reduction
    weekend_factor = 0.7 * df['is_weekend'] + 1.0 * (1 - df['is_weekend'])
    
    # Combine all factors for energy consumption
    df['energy_consumption'] = (base_load + business_load + heating_load + cooling_load) * \
                               seasonal_factor * time_factor * weekend_factor
    
    # Add random noise to make it more realistic
    df['energy_consumption'] = df['energy_consumption'] * (1 + np.random.normal(0, 0.05, n_samples))
    
    # Add humidity
    base_humidity = 50 + 20 * np.sin(2 * np.pi * (df.index.dayofyear / 365 + 0.5))
    humidity_noise = np.random.normal(0, 5, n_samples)
    df['humidity'] = base_humidity + humidity_noise
    df['humidity'] = df['humidity'].clip(20, 95)  # Realistic range
    
    # Add cloud cover (0-100%)
    df['cloud_cover'] = np.random.beta(2, 3, n_samples) * 100
    
    # Add wind speed (m/s)
    df['wind_speed'] = np.random.gamma(2, 2, n_samples)
    
    # Add occupancy
    df['occupancy'] = 0
    
    # Weekday occupancy pattern
    weekday_mask = (df['day_of_week'] < 5)
    morning_arrival = (df['hour'] >= 7) & (df['hour'] < 10) & weekday_mask
    working_hours = (df['hour'] >= 10) & (df['hour'] < 17) & weekday_mask
    evening_departure = (df['hour'] >= 17) & (df['hour'] <= 19) & weekday_mask
    
    df.loc[morning_arrival, 'occupancy'] = 0.7 * (1 + 0.3 * np.random.random(sum(morning_arrival)))
    df.loc[working_hours, 'occupancy'] = 0.9 * (1 + 0.1 * np.random.random(sum(working_hours)))
    df.loc[evening_departure, 'occupancy'] = 0.5 * (1 + 0.3 * np.random.random(sum(evening_departure)))
    
    # Weekend occupancy (much lower)
    weekend_hours = (df['hour'] >= 10) & (df['hour'] <= 15) & df['is_weekend'].astype(bool)
    df.loc[weekend_hours, 'occupancy'] = 0.2 * (1 + 0.5 * np.random.random(sum(weekend_hours)))
    
    # Ensure values are realistic and positive
    df['energy_consumption'] = df['energy_consumption'].clip(lower=10)
    
    # Add some missing values to simulate real-world data issues
    for col in ['outdoor_temperature', 'humidity', 'wind_speed']:
        missing_idx = random.sample(range(n_samples), int(n_samples * 0.01))  # 1% missing data
        df.loc[df.index[missing_idx], col] = np.nan
    
    # Round values to make them more realistic
    df['energy_consumption'] = df['energy_consumption'].round(2)
    df['outdoor_temperature'] = df['outdoor_temperature'].round(1)
    df['humidity'] = df['humidity'].round(1)
    df['wind_speed'] = df['wind_speed'].round(1)
    df['cloud_cover'] = df['cloud_cover'].round(1)
    
    # Keep only the columns needed for the regression task
    final_df = df[['energy_consumption', 'outdoor_temperature', 'humidity', 
                  'wind_speed', 'cloud_cover', 'occupancy']]
    
    return final_df

# Try to load the dataset, or generate synthetic data if file doesn't exist
try:
    # Attempt to load existing data file
    print("Attempting to load building_energy_data.csv...")
    energy_data = pd.read_csv('building_energy_data.csv', parse_dates=['timestamp'])
    print("File found and loaded successfully!")
except FileNotFoundError:
    # If file not found, generate synthetic data
    print("File not found. Generating synthetic data...")
    import random  # Import here for the synthetic data generation
    energy_data = generate_synthetic_energy_data()
    energy_data = energy_data.reset_index()  # Convert index to column for consistent format
    # Save the synthetic data
    energy_data.to_csv('building_energy_data.csv', index=False)
    print("Synthetic data generated and saved to building_energy_data.csv")

# Set timestamp as index
energy_data.set_index('timestamp', inplace=True)

# Display basic information
print(f"\nDataset shape: {energy_data.shape}")
print(f"Date range: {energy_data.index.min()} to {energy_data.index.max()}")
print(f"Missing values: {energy_data.isnull().sum().sum()}")

# Preview the data
print("\nData preview:")
print(energy_data.head())

# Summary statistics
print("\nSummary statistics:")
print(energy_data.describe())

# Fill missing values with appropriate methods
print("\nFilling missing values...")
energy_data['outdoor_temperature'].fillna(method='ffill', inplace=True)
energy_data['humidity'].fillna(method='ffill', inplace=True)
energy_data['wind_speed'].fillna(method='ffill', inplace=True)
if 'cloud_cover' in energy_data.columns:
    energy_data['cloud_cover'].fillna(method='ffill', inplace=True)
if 'occupancy' in energy_data.columns:
    energy_data['occupancy'].fillna(0, inplace=True)

# Check for any remaining missing values
print(f"Remaining missing values: {energy_data.isnull().sum().sum()}")

### Exploratory Data Analysis

In [ ]:
# Resample data to different time intervals
daily = energy_data['energy_consumption'].resample('D').mean()
weekly = energy_data['energy_consumption'].resample('W').mean()
monthly = energy_data['energy_consumption'].resample('M').mean()

# Plot time series at different intervals
plt.figure(figsize=(15, 12))

plt.subplot(3, 1, 1)
daily.plot()
plt.title('Daily Energy Consumption')
plt.ylabel('kWh')

plt.subplot(3, 1, 2)
weekly.plot()
plt.title('Weekly Energy Consumption')
plt.ylabel('kWh')

plt.subplot(3, 1, 3)
monthly.plot()
plt.title('Monthly Energy Consumption')
plt.ylabel('kWh')

plt.tight_layout()
plt.show()

# Analyze seasonality: hourly patterns
plt.figure(figsize=(12, 6))
hourly_avg = energy_data.groupby(energy_data.index.hour)['energy_consumption'].mean()
hourly_avg.plot(kind='bar')
plt.title('Average Energy Consumption by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Average kWh')
plt.xticks(rotation=0)
plt.show()

# Analyze seasonality: daily patterns
plt.figure(figsize=(12, 6))
daily_avg = energy_data.groupby(energy_data.index.dayofweek)['energy_consumption'].mean()
daily_avg.plot(kind='bar')
plt.title('Average Energy Consumption by Day of Week')
plt.xlabel('Day (0=Monday, 6=Sunday)')
plt.ylabel('Average kWh')
plt.xticks(rotation=0)
plt.show()

# Analyze relationship with temperature
plt.figure(figsize=(12, 6))
plt.scatter(energy_data['outdoor_temperature'], energy_data['energy_consumption'], alpha=0.5)
plt.title('Energy Consumption vs. Outdoor Temperature')
plt.xlabel('Outdoor Temperature (°C)')
plt.ylabel('Energy Consumption (kWh)')
plt.show()

# Calculate correlation matrix
plt.figure(figsize=(10, 8))
correlation = energy_data.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

### Feature Engineering

In [ ]:
# Create a copy of the dataframe
df = energy_data.copy()

# Temporal features
df['hour'] = df.index.hour
df['day'] = df.index.day
df['month'] = df.index.month
df['day_of_week'] = df.index.dayofweek
df['day_of_year'] = df.index.dayofyear
df['week_of_year'] = df.index.isocalendar().week
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# We'll use a simplified holiday list for the US 
# For a real project, use the holidays library
us_holidays = [
    '2020-01-01', '2020-01-20', '2020-02-17', '2020-05-25', '2020-07-03', '2020-09-07', 
    '2020-10-12', '2020-11-11', '2020-11-26', '2020-12-25',
    '2021-01-01', '2021-01-18', '2021-02-15', '2021-05-31', '2021-07-05', '2021-09-06', 
    '2021-10-11', '2021-11-11', '2021-11-25', '2021-12-24',
    '2022-01-01', '2022-01-17', '2022-02-21', '2022-05-30', '2022-07-04', '2022-09-05', 
    '2022-10-10', '2022-11-11', '2022-11-24', '2022-12-26'
]
df['is_holiday'] = df.index.strftime('%Y-%m-%d').isin(us_holidays).astype(int)

# Time of day categories
df['time_of_day'] = pd.cut(
    df['hour'], 
    bins=[0, 6, 12, 18, 24], 
    labels=['night', 'morning', 'afternoon', 'evening'],
    include_lowest=True
)
df = pd.get_dummies(df, columns=['time_of_day'])

# Cyclical encoding for hour, day of week, month
def encode_cyclical(df, col, max_val):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

df = encode_cyclical(df, 'hour', 24)
df = encode_cyclical(df, 'day_of_week', 7)
df = encode_cyclical(df, 'month', 12)

# Lag features (previous hours)
for i in [1, 2, 3, 6, 12, 24]:
    df[f'energy_lag_{i}'] = df['energy_consumption'].shift(i)

# Rolling mean features
for window in [3, 6, 12, 24]:
    df[f'rolling_mean_{window}h'] = df['energy_consumption'].rolling(window=window).mean()
    
# Temperature features
df['temp_squared'] = df['outdoor_temperature'] ** 2  # For non-linear relationships
df['heating_degree'] = np.maximum(18.5 - df['outdoor_temperature'], 0)  # Heating threshold 18.5°C
df['cooling_degree'] = np.maximum(df['outdoor_temperature'] - 21, 0)   # Cooling threshold 21°C

# Drop missing values created by lag and rolling features
df.dropna(inplace=True)

print(f"Original dataset shape: {energy_data.shape}")
print(f"Processed dataset shape: {df.shape}")

### Model Development

In [ ]:
# Define features and target variable
X = df.drop('energy_consumption', axis=1)
y = df['energy_consumption']

# Split the data into training and testing sets using a time-based split
train_end = int(len(df) * 0.8)
X_train, X_test = X[:train_end], X[train_end:]
y_train, y_test = y[:train_end], y[train_end:]

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

# Function to evaluate regression models
def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    print(f"{model_name} Performance:")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAPE: {mape:.2f}%\n")
    
    return {
        'model': model_name,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape
    }

# Store results for comparison
results = []

# XGBoost Model
print("Training XGBoost model...")
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
results.append(evaluate_model(y_test, xgb_pred, "XGBoost"))

# Compare model performance
results_df = pd.DataFrame(results)
print("Model Performance:")
print(results_df)

### Model Interpretation and Feature Importance

In [ ]:
# Plot feature importance for XGBoost model
plt.figure(figsize=(14, 10))
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Top 20 features
top_features = feature_importance.head(20)
sns.barplot(x='Importance', y='Feature', data=top_features)
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

# Plot predictions vs actual values
plt.figure(figsize=(16, 6))
test_dates = df.index[train_end:]
plt.plot(test_dates, y_test, label='Actual', alpha=0.7)
plt.plot(test_dates, xgb_pred, label='XGBoost', alpha=0.7)
plt.title('Energy Consumption: Actual vs Predicted')
plt.xlabel('Date')
plt.ylabel('Energy Consumption (kWh)')
plt.legend()
plt.tight_layout()
plt.show()

### Forecasting Future Energy Consumption

In [ ]:
# Generate predictions for the next 7 days (assuming hourly data)
last_date = df.index[-1]
forecast_horizon = 7 * 24  # 7 days of hourly predictions

# Create a dataframe with future dates for forecasting
future_dates = pd.date_range(
    start=last_date + pd.Timedelta(hours=1),
    periods=forecast_horizon,
    freq='H'
)

# Create future features
future_df = pd.DataFrame(index=future_dates)
future_df['hour'] = future_df.index.hour
future_df['day'] = future_df.index.day
future_df['month'] = future_df.index.month
future_df['day_of_week'] = future_df.index.dayofweek
future_df['day_of_year'] = future_df.index.dayofyear
future_df['week_of_year'] = future_df.index.isocalendar().week
future_df['is_weekend'] = future_df['day_of_week'].isin([5, 6]).astype(int)

# Simplified holidays for future dates
future_holidays = [
    '2023-01-01', '2023-01-16', '2023-02-20', '2023-05-29', '2023-07-04', 
    '2023-09-04', '2023-10-09', '2023-11-11', '2023-11-23', '2023-12-25'
]
future_df['is_holiday'] = future_df.index.strftime('%Y-%m-%d').isin(future_holidays).astype(int)

# Time of day categories
future_df['time_of_day'] = pd.cut(
    future_df['hour'], 
    bins=[0, 6, 12, 18, 24], 
    labels=['night', 'morning', 'afternoon', 'evening'],
    include_lowest=True
)
future_df = pd.get_dummies(future_df, columns=['time_of_day'])

# Cyclical encoding
future_df = encode_cyclical(future_df, 'hour', 24)
future_df = encode_cyclical(future_df, 'day_of_week', 7)
future_df = encode_cyclical(future_df, 'month', 12)

# Weather features (here we use averages by hour from historical data)
hourly_temp_avg = df.groupby('hour')['outdoor_temperature'].mean()
future_df['outdoor_temperature'] = future_df['hour'].map(hourly_temp_avg)
future_df['temp_squared'] = future_df['outdoor_temperature'] ** 2
future_df['heating_degree'] = np.maximum(18.5 - future_df['outdoor_temperature'], 0)
future_df['cooling_degree'] = np.maximum(future_df['outdoor_temperature'] - 21, 0)

# Add other weather features if they were in the original dataset
if 'humidity' in df.columns:
    hourly_humidity_avg = df.groupby('hour')['humidity'].mean()
    future_df['humidity'] = future_df['hour'].map(hourly_humidity_avg)

if 'wind_speed' in df.columns:
    hourly_wind_avg = df.groupby('hour')['wind_speed'].mean()
    future_df['wind_speed'] = future_df['hour'].map(hourly_wind_avg)

if 'cloud_cover' in df.columns:
    hourly_cloud_avg = df.groupby('hour')['cloud_cover'].mean()
    future_df['cloud_cover'] = future_df['hour'].map(hourly_cloud_avg)

if 'occupancy' in df.columns:
    # Similar occupancy patterns as in historical data
    future_df['occupancy'] = 0
    
    # Weekday occupancy pattern
    weekday_mask = (future_df['day_of_week'] < 5)
    morning_arrival = (future_df['hour'] >= 7) & (future_df['hour'] < 10) & weekday_mask
    working_hours = (future_df['hour'] >= 10) & (future_df['hour'] < 17) & weekday_mask
    evening_departure = (future_df['hour'] >= 17) & (future_df['hour'] <= 19) & weekday_mask
    
    future_df.loc[morning_arrival, 'occupancy'] = df.loc[df['day_of_week'] < 5].loc[
        (df['hour'] >= 7) & (df['hour'] < 10), 'occupancy'].mean()
    future_df.loc[working_hours, 'occupancy'] = df.loc[df['day_of_week'] < 5].loc[
        (df['hour'] >= 10) & (df['hour'] < 17), 'occupancy'].mean()
    future_df.loc[evening_departure, 'occupancy'] = df.loc[df['day_of_week'] < 5].loc[
        (df['hour'] >= 17) & (df['hour'] <= 19), 'occupancy'].mean()
    
    # Weekend occupancy (much lower)
    weekend_hours = (future_df['hour'] >= 10) & (future_df['hour'] <= 15) & future_df['is_weekend'].astype(bool)
    future_df.loc[weekend_hours, 'occupancy'] = df.loc[df['is_weekend'] == 1].loc[
        (df['hour'] >= 10) & (df['hour'] <= 15), 'occupancy'].mean()

# For lag features and rolling means, use the last known values from historical data
for i in [1, 2, 3, 6, 12, 24]:
    last_values = df['energy_consumption'].iloc[-i:].values
    future_df[f'energy_lag_{i}'] = np.nan
    future_df[f'energy_lag_{i}'].iloc[0] = last_values[-1]

# Initial rolling mean values
for window in [3, 6, 12, 24]:
    future_df[f'rolling_mean_{window}h'] = df['energy_consumption'].iloc[-window:].mean()

### Visualization and Analysis Code
To visualize and analyze your forecasted future energy consumption, here's a comprehensive Python code snippet using Matplotlib, Seaborn, and Plotly to generate various types of plots and data tables. This includes:
- Line charts
- Bar plots
- Heatmaps
- Time series decomposition (if applicable)
- Data tables with descriptive statistics
- Correlation matrix

You can modify this to match your prediction model (assuming future_df['predicted_energy'] will contain your model’s output).

✅ Optional: Add Real Forecast Column
- If you use a model like XGBoost, LightGBM, or a neural network to generate predictions:
```python
# Example with scikit-learn model
model = trained_model
feature_cols = [...]  # whatever features you used
future_df['predicted_energy'] = model.predict(future_df[feature_cols])
```

In [ ]:
import plotly.express as px

# Simulate predictions if not present
if 'predicted_energy' not in future_df:
    np.random.seed(42)
    future_df['predicted_energy'] = np.random.normal(loc=100, scale=10, size=len(future_df))

# --- 1. Line Plot ---
plt.figure(figsize=(15, 5))
plt.plot(future_df.index, future_df['predicted_energy'], label='Forecasted Energy', color='blue')
plt.title('Forecasted Hourly Energy Consumption (Next 7 Days)')
plt.xlabel('Datetime')
plt.ylabel('Energy Consumption')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# --- 2. Daily Forecast (Bar Plot) ---
daily_forecast = future_df['predicted_energy'].resample('D').sum()
daily_forecast.plot(kind='bar', figsize=(12, 6), color='orange')
plt.title('Daily Total Energy Consumption Forecast')
plt.xlabel('Day')
plt.ylabel('Total Energy (kWh)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- 3. Heatmap by Hour & Day ---
pivot = future_df.copy()
pivot['date'] = pivot.index.date
pivot['hour'] = pivot.index.hour
heatmap_data = pivot.pivot_table(index='hour', columns='date', values='predicted_energy')
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd')
plt.title('Hourly Energy Consumption Forecast Heatmap')
plt.xlabel('Date')
plt.ylabel('Hour of Day')
plt.tight_layout()
plt.show()

# --- 4. Average by Hour ---
hourly_avg = future_df.groupby(future_df.index.hour)['predicted_energy'].mean()
hourly_avg.plot(kind='line', marker='o', figsize=(10, 5))
plt.title('Average Forecasted Energy Consumption by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Energy Consumption')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- 5. Improved Correlation Matrix ---
plt.figure(figsize=(16, 12))
corr = future_df.select_dtypes(include=np.number).corr()

# Optional: mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            annot_kws={"size": 10}, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix (Forecast)', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# --- 6. Interactive Plot with Plotly ---
fig = px.line(future_df, x=future_df.index, y='predicted_energy',
              title='Interactive Forecasted Energy Consumption (Hourly)',
              labels={'predicted_energy': 'Energy Consumption'})
fig.update_layout(template='plotly_white')
fig.update_xaxes(title='Datetime')
fig.update_yaxes(title='Predicted Energy')
fig.show()

# --- 7. Top 20 Highest Forecasted Hours ---
print("\n🔍 Top 20 Hours with Highest Predicted Energy:")
display(future_df[['predicted_energy']].sort_values(by='predicted_energy', ascending=False).head(20))

# --- 8. Descriptive Stats Table ---
print("\n📊 Descriptive Statistics for Forecasted Energy:")
print(future_df['predicted_energy'].describe())

# --- 9. Boxplot by Day of Week ---
future_df['day_of_week_name'] = future_df.index.day_name()
plt.figure(figsize=(10, 6))
sns.boxplot(x='day_of_week_name', y='predicted_energy', data=future_df,
            order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
plt.title('Energy Forecast Distribution by Day of Week')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

# === Saving Setup ===
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"forecast_output_{timestamp}"
os.makedirs(output_dir, exist_ok=True)
pdf = PdfPages(os.path.join(output_dir, "summary_plots.pdf"))

# Simulate predictions if not present
if 'predicted_energy' not in future_df:
    np.random.seed(42)
    future_df['predicted_energy'] = np.random.normal(loc=100, scale=10, size=len(future_df))

# --- 1. Line Plot ---
plt.figure(figsize=(15, 5))
plt.plot(future_df.index, future_df['predicted_energy'], label='Forecasted Energy', color='blue')
plt.title('Forecasted Hourly Energy Consumption (Next 7 Days)')
plt.xlabel('Datetime')
plt.ylabel('Energy Consumption')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "01_lineplot.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "01_lineplot.svg"))
pdf.savefig()
plt.show()

# --- 2. Daily Forecast (Bar Plot) ---
daily_forecast = future_df['predicted_energy'].resample('D').sum()
daily_forecast.plot(kind='bar', figsize=(12, 6), color='orange')
plt.title('Daily Total Energy Consumption Forecast')
plt.xlabel('Day')
plt.ylabel('Total Energy (kWh)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "02_daily_bar.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "02_daily_bar.svg"))
pdf.savefig()
plt.show()

# --- 3. Heatmap by Hour & Day ---
pivot = future_df.copy()
pivot['date'] = pivot.index.date
pivot['hour'] = pivot.index.hour
heatmap_data = pivot.pivot_table(index='hour', columns='date', values='predicted_energy')
plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd')
plt.title('Hourly Energy Consumption Forecast Heatmap')
plt.xlabel('Date')
plt.ylabel('Hour of Day')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "03_heatmap.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "03_heatmap.svg"))
pdf.savefig()
plt.show()

# --- 4. Average by Hour ---
hourly_avg = future_df.groupby(future_df.index.hour)['predicted_energy'].mean()
hourly_avg.plot(kind='line', marker='o', figsize=(10, 5))
plt.title('Average Forecasted Energy Consumption by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Energy Consumption')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "04_avg_hourly.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "04_avg_hourly.svg"))
pdf.savefig()
plt.show()

# --- 5. Improved Correlation Matrix ---
plt.figure(figsize=(16, 12))
corr = future_df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            annot_kws={"size": 10}, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix (Forecast)', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "05_corr_matrix.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "05_corr_matrix.svg"))
pdf.savefig()
plt.show()

# --- 6. Interactive Plot with Plotly ---
fig = px.line(future_df, x=future_df.index, y='predicted_energy',
              title='Interactive Forecasted Energy Consumption (Hourly)',
              labels={'predicted_energy': 'Energy Consumption'})
fig.update_layout(template='plotly_white')
fig.update_xaxes(title='Datetime')
fig.update_yaxes(title='Predicted Energy')
fig.write_image(os.path.join(output_dir, "06_plotly_interactive.png"), scale=2)
fig.write_image(os.path.join(output_dir, "06_plotly_interactive.svg"))
fig.show()

# --- 7. Top 20 Highest Forecasted Hours ---
print("\n🔍 Top 20 Hours with Highest Predicted Energy:")
display(future_df[['predicted_energy']].sort_values(by='predicted_energy', ascending=False).head(20))

# --- 8. Descriptive Stats Table ---
print("\n📊 Descriptive Statistics for Forecasted Energy:")
print(future_df['predicted_energy'].describe())

# --- 9. Boxplot by Day of Week ---
future_df['day_of_week_name'] = future_df.index.day_name()
plt.figure(figsize=(10, 6))
sns.boxplot(x='day_of_week_name', y='predicted_energy', data=future_df,
            order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
plt.title('Energy Forecast Distribution by Day of Week')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "09_boxplot_weekday.png"), dpi=300)
plt.savefig(os.path.join(output_dir, "09_boxplot_weekday.svg"))
pdf.savefig()
plt.show()

# --- Finish PDF ---
pdf.close()
print(f"\n✅ All plots saved in: {output_dir}")

### Export forecast results to CSV and Excel

In [ ]:
# --- 10. Export forecast results to CSV and Excel ---

# Select relevant columns to export (optional: include only important features)
export_cols = ['predicted_energy']
if 'day_of_week_name' in future_df.columns:
    export_cols.append('day_of_week_name')

# Create a copy with datetime as column (optional for Excel readability)
export_df = future_df[export_cols].copy()
export_df.reset_index(inplace=True)
export_df.rename(columns={'index': 'datetime'}, inplace=True)

# Save to CSV
csv_path = 'forecast_energy_results.csv'
export_df.to_csv(csv_path, index=False)
print(f"✅ Forecast results saved to: {csv_path}")

# Save to Excel
excel_path = 'forecast_energy_results.xlsx'
export_df.to_excel(excel_path, index=False)
print(f"✅ Forecast results saved to: {excel_path}")

In [ ]:
# --- 11. Export ALL features to CSV and Excel ---

# Make a copy and reset index for datetime to be a column
full_export_df = future_df.copy()
full_export_df.reset_index(inplace=True)
full_export_df.rename(columns={'index': 'datetime'}, inplace=True)

# Save full features to CSV
full_csv_path = 'forecast_full_features.csv'
full_export_df.to_csv(full_csv_path, index=False)
print(f"✅ Full feature forecast saved to CSV: {full_csv_path}")

# Save full features to Excel
full_excel_path = 'forecast_full_features.xlsx'
full_export_df.to_excel(full_excel_path, index=False)
print(f"✅ Full feature forecast saved to Excel: {full_excel_path}")

### Interactive Forecast Dashboard with Widgets (Jupyter)

This section provides an **interactive dashboard** to explore energy consumption forecasts using `ipywidgets`.  
You can:
- Select forecast duration (in days)
- Choose which plots to display
- Export to CSV/Excel
- Simulate uploads to cloud platforms

---

✅ **Features**

- **Forecast days slider**: Set forecast horizon (1–30 days)
- **Plot selector**: Choose from:
  - Line Plot
  - Daily Boxplot
  - Weekly Average
  - Hourly Pattern
  - Feature Correlation
- **Export Button**: Save results to local `.csv` and `.xlsx`
- **Upload Button**: Placeholder for cloud sync (Google Drive, Dropbox, OneDrive, GitHub)

---

🧩 **Required Libraries**

```python
!pip install ipywidgets pandas matplotlib plotly openpyxl
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets
forecast_days_slider = widgets.IntSlider(value=7, min=1, max=30, step=1, description='Forecast Days:')
plot_selector = widgets.SelectMultiple(
    options=['Line Plot', 'Daily Boxplot', 'Weekly Average', 'Hourly Pattern', 'Feature Correlation'],
    value=['Line Plot'], description='Plots:', layout={'height': '120px'}
)
export_button = widgets.Button(description="Export to CSV & Excel", button_style='success')
upload_button = widgets.Button(description="Upload to Cloud", button_style='info')
output_area = widgets.Output()

# Forecast + Visualizations
def run_forecast(change=None):
    with output_area:
        clear_output(wait=True)
        forecast_horizon = forecast_days_slider.value * 24
        future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(hours=1), periods=forecast_horizon, freq='H')
        future_df = pd.DataFrame(index=future_dates)
        future_df['hour'] = future_df.index.hour
        future_df['day_of_week'] = future_df.index.dayofweek
        future_df['is_weekend'] = future_df['day_of_week'].isin([5, 6]).astype(int)
        future_df['outdoor_temperature'] = df.groupby('hour')['outdoor_temperature'].mean().reindex(future_df['hour']).values
        future_df['predicted_energy'] = (
            100 + 5 * np.sin(future_df['hour'] / 24 * 2 * np.pi) +
            3 * future_df['is_weekend'] +
            0.5 * (18.5 - future_df['outdoor_temperature']) +
            np.random.normal(0, 2, size=len(future_df))
        )
        globals()['future_df'] = future_df

        # Line Plot
        if 'Line Plot' in plot_selector.value:
            fig = px.line(future_df, x=future_df.index, y='predicted_energy', title="Forecasted Energy Consumption")
            fig.show()

        # Daily Boxplot
        if 'Daily Boxplot' in plot_selector.value:
            future_df['date'] = future_df.index.date
            plt.figure(figsize=(12, 4))
            future_df.boxplot(column='predicted_energy', by='date')
            plt.title("Daily Energy Forecast Spread")
            plt.suptitle("")
            plt.xticks(rotation=45)
            plt.show()

        # Weekly Average
        if 'Weekly Average' in plot_selector.value:
            plt.figure(figsize=(10, 4))
            future_df.groupby(future_df.index.dayofweek)['predicted_energy'].mean().plot(kind='bar')
            plt.title("Average Energy Forecast by Day of Week")
            plt.xticks(ticks=range(7), labels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
            plt.ylabel("Energy")
            plt.show()

        # Hourly Pattern
        if 'Hourly Pattern' in plot_selector.value:
            plt.figure(figsize=(10, 4))
            future_df.groupby(future_df.index.hour)['predicted_energy'].mean().plot()
            plt.title("Average Energy Forecast by Hour")
            plt.xlabel("Hour of Day")
            plt.ylabel("Energy")
            plt.grid(True)
            plt.show()

        # Feature Correlation
        if 'Feature Correlation' in plot_selector.value:
            corr = future_df.corr(numeric_only=True)
            plt.figure(figsize=(12, 10))
            plt.imshow(corr, cmap='coolwarm', interpolation='none')
            plt.colorbar(label='Correlation')
            plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
            plt.yticks(range(len(corr.index)), corr.index, fontsize=8)
            plt.title("Feature Correlation Matrix")
            plt.tight_layout()
            plt.show()

# Export logic
def export_forecast(button=None):
    if 'future_df' not in globals():
        print("⛔ Run forecast first.")
        return
    df_out = future_df.copy().reset_index().rename(columns={'index': 'datetime'})
    df_out.to_csv("interactive_forecast.csv", index=False)
    df_out.to_excel("interactive_forecast.xlsx", index=False)
    print("✅ Exported: interactive_forecast.csv / interactive_forecast.xlsx")

# Upload stub logic
def upload_to_cloud(button=None):
    print("☁️ Stub: Upload logic for Google Drive, Dropbox, OneDrive, GitHub goes here.")

# Bind events
forecast_days_slider.observe(run_forecast, names='value')
plot_selector.observe(run_forecast, names='value')
export_button.on_click(export_forecast)
upload_button.on_click(upload_to_cloud)

# Render dashboard
display(widgets.VBox([
    widgets.HBox([forecast_days_slider, export_button, upload_button]),
    plot_selector,
    output_area
]))
run_forecast()

### Upload Forecast Files to Cloud Platforms
how to upload the forecast files to:
- Google Drive
- Dropbox
- OneDrive
- GitHub

✅ Notes:
- You'll need authentication tokens or credentials for each platform
- For Google Drive, you’ll need to authorize via browser (1-time).
- For Dropbox, OneDrive, and GitHub, you need access tokens (store securely).
- You can load these with os.getenv() using a .env file and python-dotenv.

#### 1. Upload to Google Drive (via pydrive)

In [ ]:
# # Install: pip install pydrive
# from pydrive.auth import GoogleAuth
# from pydrive.drive import GoogleDrive

# # Authenticate
# gauth = GoogleAuth()
# gauth.LocalWebserverAuth()
# drive = GoogleDrive(gauth)

# # Upload files
# for file_path in ['forecast_energy_results.csv', 'forecast_energy_results.xlsx',
#                   'forecast_full_features.csv', 'forecast_full_features.xlsx']:
#     upload_file = drive.CreateFile({'title': file_path})
#     upload_file.SetContentFile(file_path)
#     upload_file.Upload()
#     print(f"✅ Uploaded to Google Drive: {file_path}")

#### 2. Upload to Dropbox (via dropbox SDK)

In [ ]:
# # Install: pip install dropbox
# import dropbox
# import os

# DROPBOX_ACCESS_TOKEN = os.getenv("DROPBOX_ACCESS_TOKEN")  # Set this in your .env

# dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)

# for file_path in ['forecast_energy_results.csv', 'forecast_energy_results.xlsx',
#                   'forecast_full_features.csv', 'forecast_full_features.xlsx']:
#     with open(file_path, "rb") as f:
#         dbx.files_upload(f.read(), f'/{file_path}', mode=dropbox.files.WriteMode("overwrite"))
#         print(f"✅ Uploaded to Dropbox: {file_path}")

#### 3. Upload to OneDrive (via msal + Microsoft Graph API)
This is more complex; here is the basic structure:

In [ ]:
# # Install: pip install msal requests
# import os
# import requests
# from msal import PublicClientApplication

# CLIENT_ID = os.getenv("ONEDRIVE_CLIENT_ID")
# AUTHORITY = "https://login.microsoftonline.com/common"
# SCOPES = ["Files.ReadWrite.All"]

# app = PublicClientApplication(CLIENT_ID, authority=AUTHORITY)
# result = None

# # Interactive login
# accounts = app.get_accounts()
# if accounts:
#     result = app.acquire_token_silent(SCOPES, account=accounts[0])
# if not result:
#     result = app.acquire_token_interactive(SCOPES)

# access_token = result['access_token']

# # Upload files
# headers = {'Authorization': f'Bearer {access_token}'}
# upload_url = "https://graph.microsoft.com/v1.0/me/drive/root:/{}:/content"

# for file_path in ['forecast_energy_results.csv', 'forecast_energy_results.xlsx',
#                   'forecast_full_features.csv', 'forecast_full_features.xlsx']:
#     with open(file_path, 'rb') as f:
#         resp = requests.put(upload_url.format(file_path), headers=headers, data=f)
#         if resp.status_code == 201 or resp.status_code == 200:
#             print(f"✅ Uploaded to OneDrive: {file_path}")
#         else:
#             print(f"❌ OneDrive Upload Failed: {file_path}")

#### 4. Upload to GitHub (via PyGithub)

In [ ]:
# # Install: pip install PyGithub
# from github import Github
# import base64

# GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
# REPO_NAME = "your-username/your-repo"  # Format: user/repo
# TARGET_BRANCH = "main"

# g = Github(GITHUB_TOKEN)
# repo = g.get_repo(REPO_NAME)

# for file_path in ['forecast_energy_results.csv', 'forecast_energy_results.xlsx',
#                   'forecast_full_features.csv', 'forecast_full_features.xlsx']:
#     with open(file_path, 'rb') as f:
#         content = f.read()
#     try:
#         # Check if file exists
#         existing_file = repo.get_contents(file_path, ref=TARGET_BRANCH)
#         repo.update_file(existing_file.path, f"Update {file_path}", content.decode(), existing_file.sha, branch=TARGET_BRANCH)
#         print(f"✅ Updated file on GitHub: {file_path}")
#     except:
#         repo.create_file(file_path, f"Add {file_path}", content.decode(), branch=TARGET_BRANCH)
#         print(f"✅ Uploaded new file to GitHub: {file_path}")

### Gradio App Code for Forecasted Energy Data
Gradio app that lets users interactively explore forecasted energy consumption using sliders and dropdowns. It includes:
- Date/time selection
- Filter by time of day or weather conditions
- Plot energy predictions
- Show data table

In [ ]:
# !pip install gradio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime

# Load forecast data
# forecast_df = pd.read_csv("forecast_energy_results.csv", parse_dates=["datetime"])
forecast_df = pd.read_csv("forecast_full_features.csv", parse_dates=["datetime"])


# --- Identify the temperature column ---
temperature_col = None
for col in forecast_df.columns:
    if "temp" in col.lower():
        temperature_col = col
        break

if not temperature_col:
    raise ValueError("Temperature column not found! Please check the dataset.")

# Time of day mapping
time_of_day_columns = {
    "night": "time_of_day_night",
    "morning": "time_of_day_morning",
    "afternoon": "time_of_day_afternoon",
    "evening": "time_of_day_evening"
}

# Plotting function
def forecast_plot(start_date, end_date, time_of_day, min_temp, max_temp):
    try:
        df = forecast_df.copy()

        # Convert dates
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)

        if start > end:
            raise ValueError("Start date must be before end date.")

        # Filter by date
        df = df[(df["datetime"] >= start) & (df["datetime"] <= end)]

        # Time of day
        if time_of_day != "All":
            col = time_of_day_columns.get(time_of_day.lower())
            if col and col in df.columns:
                df = df[df[col] == 1]
            else:
                raise ValueError(f"Time of day column '{col}' not found.")

        # Filter by temperature
        df = df[(df[temperature_col] >= min_temp) & (df[temperature_col] <= max_temp)]

        if df.empty:
            raise ValueError("No data after filtering. Try different settings.")

        # Plot
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(df["datetime"], df["predicted_energy"], label="Predicted", color="green")
        ax.set_title("Forecasted Energy Consumption")
        ax.set_xlabel("Datetime")
        ax.set_ylabel("Energy (kWh)")
        ax.grid(True)
        fig.autofmt_xdate()

        # Preview
        preview_df = df[["datetime", "predicted_energy", temperature_col]].copy()
        preview_df["datetime"] = preview_df["datetime"].dt.strftime("%Y-%m-%d %H:%M")
        preview = preview_df.head(20).to_string(index=False)

        return fig, preview

    except Exception as e:
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "⚠️ ERROR", ha='center', va='center', fontsize=14)
        ax.axis("off")
        return fig, f"⚠️ {str(e)}"

# Gradio app
demo = gr.Interface(
    fn=forecast_plot,
    inputs=[
        gr.Textbox(label="Start Date (YYYY-MM-DD)", value=str(forecast_df["datetime"].min().date())),
        gr.Textbox(label="End Date (YYYY-MM-DD)", value=str(forecast_df["datetime"].max().date())),
        gr.Dropdown(label="Time of Day", choices=["All", "night", "morning", "afternoon", "evening"], value="All"),
        gr.Slider(label="Min Temperature (°C)", minimum=-10, maximum=40, step=1, value=5),
        gr.Slider(label="Max Temperature (°C)", minimum=-10, maximum=40, step=1, value=30),
    ],
    outputs=[
        gr.Plot(label="Energy Forecast Plot"),
        gr.Textbox(label="Data Preview / Error Info")
    ],
    title="📊 Forecast Explorer",
    description="Adjust filters below. If an error occurs, it will appear in the preview box."
)

demo.launch()

#### Updated Gradio app code with the improved forecast_plot function that:
- Loads your CSV with all features
- Shows debug info about data range and filtering
- Handles missing columns gracefully
- Filters by date, temperature, and time_of_day

Returns meaningful error messages if filtering excludes all data or if something’s missing

How to use this:
- pload your CSV file (must have at least datetime and predicted_energy columns).
- Adjust start/end dates (format: YYYY-MM-DD).
- Choose time of day filter or "All".
- Adjust min and max temperature (if your CSV contains outdoor_temperature column).
- The plot and filtering info / errors appear below.

In [ ]:
# import pandas as pd
# import plotly.express as px
# import gradio as gr

# def forecast_plot(file, start_date, end_date, time_filter, min_temp, max_temp):
#     try:
#         # Load CSV
#         df = pd.read_csv(file.name, parse_dates=["datetime"])
        
#         # Check required columns
#         required_cols = ["datetime", "predicted_energy"]
#         for col in required_cols:
#             if col not in df.columns:
#                 return None, f"❌ CSV must include '{col}' column"
        
#         # Debug info - datetime range
#         min_date = df["datetime"].min().date()
#         max_date = df["datetime"].max().date()
#         msg = f"Data datetime range: {min_date} to {max_date}\n"
        
#         # Convert inputs to datetime
#         start = pd.to_datetime(start_date)
#         end = pd.to_datetime(end_date)
#         if start > end:
#             return None, "❌ Error: Start date must be before End date"
        
#         # Filter by date range
#         df = df[(df["datetime"] >= start) & (df["datetime"] <= end)]
#         if df.empty:
#             return None, msg + "❌ No data in selected date range."
        
#         # Temperature filter (if column exists)
#         if "outdoor_temperature" in df.columns:
#             if min_temp > max_temp:
#                 return None, "❌ Error: Min temperature cannot be greater than Max temperature"
#             df = df[(df["outdoor_temperature"] >= min_temp) & (df["outdoor_temperature"] <= max_temp)]
#             msg += f"Filtered by temperature: {min_temp} to {max_temp} °C\n"
#             if df.empty:
#                 return None, msg + "❌ No data after temperature filtering."
        
#         # Time of day filter
#         if time_filter != "All":
#             col_name = f"time_of_day_{time_filter}"
#             if col_name in df.columns:
#                 df = df[df[col_name] == 1]
#                 msg += f"Filtered by time_of_day: {time_filter}\n"
#                 if df.empty:
#                     return None, msg + "❌ No data after time_of_day filtering."
#             else:
#                 return None, f"❌ Column '{col_name}' not found in CSV for time_of_day filtering."
        
#         # Plot forecast
#         fig = px.line(df, x="datetime", y="predicted_energy",
#                       title="📈 Forecasted Energy Consumption",
#                       labels={"predicted_energy": "Energy (kWh)"})
        
#         # Show preview of data
#         preview = df[["datetime", "predicted_energy"]].head(20).to_string(index=False)
#         msg += "\nPreview of filtered data (top 20 rows):\n" + preview
        
#         return fig, msg

#     except Exception as e:
#         return None, f"❌ ERROR: {str(e)}"

# # Gradio interface
# demo = gr.Interface(
#     fn=forecast_plot,
#     inputs=[
#         gr.File(label="Upload CSV with all features"),
#         gr.Textbox(label="Start Date (YYYY-MM-DD)", value="2023-01-01"),
#         gr.Textbox(label="End Date (YYYY-MM-DD)", value="2023-01-07"),
#         gr.Dropdown(choices=["All", "night", "morning", "afternoon", "evening"], label="Time of Day Filter", value="All"),
#         gr.Slider(minimum=-10, maximum=40, value=5, label="Min Temperature (°C)"),
#         gr.Slider(minimum=-10, maximum=40, value=30, label="Max Temperature (°C)")
#     ],
#     outputs=[
#         gr.Plot(label="Forecast Plot"),
#         gr.Textbox(label="Filtering & Data Info / Errors", lines=15)
#     ],
#     title="📊 Interactive Forecast: Energy Consumption",
#     description="Upload your forecast CSV and explore predicted energy consumption with filters."
# )

# demo.launch()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

# Mapping for time_of_day columns
time_of_day_columns = {
    "night": "time_of_day_night",
    "morning": "time_of_day_morning",
    "afternoon": "time_of_day_afternoon",
    "evening": "time_of_day_evening"
}

def forecast_plot(file, start_date, end_date, time_of_day, min_temp, max_temp):
    if file is None:
        # No file uploaded yet
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "⚠️ Please upload a CSV file.", ha='center', va='center', fontsize=14)
        ax.axis("off")
        return fig, "⚠️ Please upload a CSV file to start."

    try:
        # Load CSV and parse datetime
        df = pd.read_csv(file.name, parse_dates=["datetime"])
        
        # Check required columns
        for col in ["datetime", "predicted_energy"]:
            if col not in df.columns:
                raise ValueError(f"CSV must contain '{col}' column.")
        
        # Detect temperature column (optional)
        temperature_col = None
        for c in df.columns:
            if "temp" in c.lower():
                temperature_col = c
                break
        
        # Convert dates
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        if start > end:
            raise ValueError("Start date must be before end date.")

        # Filter by date range
        df = df[(df["datetime"] >= start) & (df["datetime"] <= end)]

        # Filter by time of day if requested
        if time_of_day != "All":
            col = time_of_day_columns.get(time_of_day.lower())
            if col:
                if col not in df.columns:
                    raise ValueError(f"Column '{col}' not found for time_of_day filtering.")
                df = df[df[col] == 1]

        # Filter by temperature if column exists
        if temperature_col:
            if min_temp > max_temp:
                raise ValueError("Min temperature cannot be greater than Max temperature.")
            df = df[(df[temperature_col] >= min_temp) & (df[temperature_col] <= max_temp)]

        if df.empty:
            raise ValueError("No data after filtering. Try different settings.")

        # Plot using matplotlib
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(df["datetime"], df["predicted_energy"], label="Predicted Energy", color="green")
        ax.set_title("Forecasted Energy Consumption")
        ax.set_xlabel("Datetime")
        ax.set_ylabel("Energy (kWh)")
        ax.grid(True)
        fig.autofmt_xdate()

        # Preview top 20 rows (datetime, predicted_energy, temperature if any)
        preview_cols = ["datetime", "predicted_energy"]
        if temperature_col:
            preview_cols.append(temperature_col)
        preview_df = df[preview_cols].copy()
        preview_df["datetime"] = preview_df["datetime"].dt.strftime("%Y-%m-%d %H:%M")
        preview = preview_df.head(20).to_string(index=False)

        info_msg = f"Data filtered: {len(df)} rows.\n"
        info_msg += f"Date range: {start.date()} to {end.date()}\n"
        if time_of_day != "All":
            info_msg += f"Time of day filter: {time_of_day}\n"
        if temperature_col:
            info_msg += f"Temperature filter ({temperature_col}): {min_temp}°C to {max_temp}°C\n"
        info_msg += "\nPreview of filtered data (top 20 rows):\n" + preview

        return fig, info_msg

    except Exception as e:
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "⚠️ ERROR\n" + str(e), ha='center', va='center', fontsize=12, color="red")
        ax.axis("off")
        return fig, f"⚠️ ERROR: {str(e)}"


# Gradio interface
demo = gr.Interface(
    fn=forecast_plot,
    inputs=[
        gr.File(label="Upload CSV with Features"),
        gr.Textbox(label="Start Date (YYYY-MM-DD)", value="2023-01-01"),
        gr.Textbox(label="End Date (YYYY-MM-DD)", value="2023-01-07"),
        gr.Dropdown(choices=["All", "night", "morning", "afternoon", "evening"], label="Time of Day Filter", value="All"),
        gr.Slider(minimum=-50, maximum=50, step=1, value=5, label="Min Temperature (°C)"),
        gr.Slider(minimum=-50, maximum=50, step=1, value=30, label="Max Temperature (°C)")
    ],
    outputs=[
        gr.Plot(label="Forecast Plot"),
        gr.Textbox(label="Info / Errors", lines=15)
    ],
    title="📊 Interactive Forecast: Energy Consumption",
    description="Upload your CSV file with forecast data and explore predicted energy consumption with filters."
)

demo.launch()

Updated, fully working code with improved warnings instead of errors, so user sees friendly info messages and no hard errors stop the UI.

Replaced exceptions with clear warning messages and always return a valid figure and info text.

- All errors replaced with warnings shown as orange text on plot + warning message box
- Added check for temperature filtering range vs actual data and shows friendly hint (not error)
- Validates date format and start/end order, but shows friendly warning if wrong
- Checks missing columns and informs user clearly
- If no data after any filter, shows a warning message on plot and info box instead of hard error
- Always returns a matplotlib figure so UI never breaks

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

# Mapping for time_of_day columns
time_of_day_columns = {
    "night": "time_of_day_night",
    "morning": "time_of_day_morning",
    "afternoon": "time_of_day_afternoon",
    "evening": "time_of_day_evening"
}

def forecast_plot(file, start_date, end_date, time_of_day, min_temp, max_temp):
    if file is None:
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "⚠️ Please upload a CSV file.", ha='center', va='center', fontsize=14)
        ax.axis("off")
        return fig, "⚠️ Please upload a CSV file to start."

    try:
        # Load CSV and parse datetime
        df = pd.read_csv(file.name, parse_dates=["datetime"])

        # Check required columns
        missing_cols = [col for col in ["datetime", "predicted_energy"] if col not in df.columns]
        if missing_cols:
            fig, ax = plt.subplots()
            ax.text(0.5, 0.5, f"⚠️ Missing required columns:\n{', '.join(missing_cols)}", ha='center', va='center', fontsize=14)
            ax.axis("off")
            return fig, f"⚠️ CSV must contain columns: {', '.join(missing_cols)}"

        # Detect temperature column (optional)
        temperature_col = None
        for c in df.columns:
            if "temp" in c.lower():
                temperature_col = c
                break

        # Convert dates
        try:
            start = pd.to_datetime(start_date)
            end = pd.to_datetime(end_date)
            if start > end:
                return _warn_fig("Start date must be before end date."), "⚠️ Start date must be before end date."
        except Exception:
            return _warn_fig("Invalid date format."), "⚠️ Please enter valid dates (YYYY-MM-DD)."

        # Filter by date range
        df = df[(df["datetime"] >= start) & (df["datetime"] <= end)]

        if df.empty:
            return _warn_fig("No data in selected date range."), "⚠️ No data found in the selected date range."

        # Filter by time of day if requested
        if time_of_day != "All":
            col = time_of_day_columns.get(time_of_day.lower())
            if col:
                if col not in df.columns:
                    return _warn_fig(f"Column '{col}' not found for time_of_day filtering."), f"⚠️ Column '{col}' not found for time_of_day filtering."
                df = df[df[col] == 1]
                if df.empty:
                    return _warn_fig(f"No data for time_of_day '{time_of_day}'."), f"⚠️ No data for time_of_day '{time_of_day}'."

        # Filter by temperature if column exists
        if temperature_col:
            if min_temp > max_temp:
                return _warn_fig("Min temperature cannot be greater than Max temperature."), "⚠️ Min temperature cannot be greater than Max temperature."

            # Check actual temperature range before filtering
            temp_min = df[temperature_col].min()
            temp_max = df[temperature_col].max()
            if max_temp < temp_min or min_temp > temp_max:
                msg = (f"No data after temperature filtering ({min_temp}°C to {max_temp}°C).\n"
                       f"Actual temperature range: {temp_min:.1f}°C to {temp_max:.1f}°C.\n"
                       "Try adjusting temperature range within actual data range.")
                return _warn_fig(msg), "⚠️ " + msg

            df = df[(df[temperature_col] >= min_temp) & (df[temperature_col] <= max_temp)]

            if df.empty:
                return _warn_fig("No data after temperature filtering."), "⚠️ No data after temperature filtering. Try adjusting temperature range."

        # Plot using matplotlib
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(df["datetime"], df["predicted_energy"], label="Predicted Energy", color="green")
        ax.set_title("Forecasted Energy Consumption")
        ax.set_xlabel("Datetime")
        ax.set_ylabel("Energy (kWh)")
        ax.grid(True)
        fig.autofmt_xdate()

        # Preview top 20 rows (datetime, predicted_energy, temperature if any)
        preview_cols = ["datetime", "predicted_energy"]
        if temperature_col:
            preview_cols.append(temperature_col)
        preview_df = df[preview_cols].copy()
        preview_df["datetime"] = preview_df["datetime"].dt.strftime("%Y-%m-%d %H:%M")
        preview = preview_df.head(20).to_string(index=False)

        info_msg = f"Data filtered: {len(df)} rows.\n"
        info_msg += f"Date range: {start.date()} to {end.date()}\n"
        if time_of_day != "All":
            info_msg += f"Time of day filter: {time_of_day}\n"
        if temperature_col:
            info_msg += f"Temperature filter ({temperature_col}): {min_temp}°C to {max_temp}°C\n"
        info_msg += "\nPreview of filtered data (top 20 rows):\n" + preview

        return fig, info_msg

    except Exception as e:
        return _warn_fig("Unexpected error occurred."), f"⚠️ Unexpected error: {str(e)}"

def _warn_fig(message):
    fig, ax = plt.subplots()
    ax.text(0.5, 0.5, "⚠️\n" + message, ha='center', va='center', fontsize=12, color="orange")
    ax.axis("off")
    return fig

# Gradio interface
demo = gr.Interface(
    fn=forecast_plot,
    inputs=[
        gr.File(label="Upload CSV with Features"),
        gr.Textbox(label="Start Date (YYYY-MM-DD)", value="2023-01-01"),
        gr.Textbox(label="End Date (YYYY-MM-DD)", value="2023-01-07"),
        gr.Dropdown(choices=["All", "night", "morning", "afternoon", "evening"], label="Time of Day Filter", value="All"),
        gr.Slider(minimum=-50, maximum=50, step=1, value=5, label="Min Temperature (°C)"),
        gr.Slider(minimum=-50, maximum=50, step=1, value=30, label="Max Temperature (°C)")
    ],
    outputs=[
        gr.Plot(label="Forecast Plot"),
        gr.Textbox(label="Info / Warnings", lines=15)
    ],
    title="📊 Interactive Forecast: Energy Consumption",
    description="Upload your CSV file with forecast data and explore predicted energy consumption with filters."
)

demo.launch()